In [1]:
import os
import re
import collections
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")
os.makedirs("eda_figures", exist_ok=True)

In [2]:
plt.rcParams.update({
    "font.family"      : "DejaVu Sans",
    "font.size"        : 11,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.titlesize"   : 13,
    "axes.titleweight" : "bold",
    "figure.dpi"       : 130,
    "savefig.dpi"      : 150,
    "savefig.bbox"     : "tight",
})

PALETTE = [
    "#2D6A9F", "#E07B39", "#3BAA6E", "#C0392B",
    "#8E44AD", "#16A085", "#F39C12", "#2C3E50",
    "#1ABC9C", "#E74C3C",
]

CSV_PATH = r"C:\Users\ddumi\Desktop\Faculty\MasterSecondYear\Dissertation\TalentCLEF\Data\A\Training\normalization\job_applicant_dataset.csv"

In [3]:
print("Loading dataset …")
df = pd.read_csv(CSV_PATH)
print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print(f"  Missing values: {df.isnull().sum().sum()}")

Loading dataset …
  Rows: 10,000  |  Columns: 9
  Missing values: 0


In [4]:
CATEGORY_MAP = {
    "Software Engineer"              : "Technology & Data",
    "Web Developer"                  : "Technology & Data",
    "Database Administrator"         : "Technology & Data",
    "Systems Analyst"                : "Technology & Data",
    "Cloud Architect"                : "Technology & Data",
    "AI Researcher"                  : "Technology & Data",
    "AI Specialist"                  : "Technology & Data",
    "Machine Learning Engineer"      : "Technology & Data",
    "Data Analyst"                   : "Technology & Data",
    "Robotics Engineer"              : "Technology & Data",
    "Cybersecurity Analyst"          : "Technology & Data",
    "UX Designer"                    : "Technology & Data",
    "Physician"                      : "Healthcare",
    "Nurse"                          : "Healthcare",
    "Pharmacist"                     : "Healthcare",
    "Dentist"                        : "Healthcare",
    "Veterinarian"                   : "Healthcare",
    "Biomedical Engineer"            : "Healthcare",
    "Psychologist"                   : "Healthcare",
    "Financial Analyst"              : "Finance & Business",
    "Accountant"                     : "Finance & Business",
    "Business Analyst"               : "Finance & Business",
    "Lawyer"                         : "Legal",
    "Legal Consultant"               : "Legal",
    "Mechanical Engineer"            : "Engineering & Architecture",
    "Civil Engineer"                 : "Engineering & Architecture",
    "Electrician"                    : "Engineering & Architecture",
    "Environmental Scientist"        : "Engineering & Architecture",
    "Urban Planner"                  : "Engineering & Architecture",
    "Construction Manager"           : "Engineering & Architecture",
    "Architect"                      : "Engineering & Architecture",
    "Operations Manager"             : "Management & Operations",
    "Supply Chain Manager"           : "Management & Operations",
    "Product Manager"                : "Management & Operations",
    "Marketing Manager"              : "Management & Operations",
    "HR Specialist"                  : "Management & Operations",
    "Sales Representative"           : "Management & Operations",
    "Customer Service Representative": "Management & Operations",
    "Graphic Designer"               : "Creative & Media",
    "Content Writer"                 : "Creative & Media",
    "Journalist"                     : "Creative & Media",
    "Creative Director"              : "Creative & Media",
    "SEO Specialist"                 : "Creative & Media",
    "Teacher"                        : "Education & Research",
    "Social Worker"                  : "Education & Research",
    "Research Scientist"             : "Education & Research",
    "Fitness Coach"                  : "Fitness & Wellness",
    "Personal Trainer"               : "Fitness & Wellness",
    "Chef"                           : "Other",
    "Event Planner"                  : "Other",
    "Pilot"                          : "Other",
}

df["Category"] = df["Job Roles"].map(CATEGORY_MAP).fillna("Other")

def parse_resume(text):
    t = str(text)

    sm = re.findall(r"SKILLS\s*\n(.*?)(?:CERTIFICATIONS|---|\Z)", t, re.DOTALL)
    skills = []
    if sm:
        skills = [
            s.strip("- ").strip()
            for s in sm[0].split("\n")
            if s.strip().startswith("-")
        ]

    yr_m = re.search(r"(\d+)-(\d+) years", t)
    yr_range = f"{yr_m.group(1)}-{yr_m.group(2)}" if yr_m else None

    deg_m = re.search(
        r"(Bachelor|Master|Doctor|PhD|Juris|Associate)[^\n]*\n", t
    )
    degree = deg_m.group(1) if deg_m else None

    full_deg_m = re.search(
        r"(Bachelor of [^\n]+|Master of [^\n]+|Doctor of [^\n]+|"
        r"PhD in [^\n]+|Juris Doctor[^\n]*|Associate[^\n]+|"
        r"Doctor of Pharmacy[^\n]*|Doctor of Medicine[^\n]*|"
        r"Bachelor of Medicine[^\n]*|Bachelor of Laws[^\n]*|"
        r"Master of Laws[^\n]*)\n", t
    )
    full_degree = full_deg_m.group(1).strip() if full_deg_m else None

    grad_m = re.search(r"Graduated: \w+ (\d{4})", t)
    grad_yr = int(grad_m.group(1)) if grad_m else None

    cert_m = re.findall(
        r"CERTIFICATIONS\s*\n(.*?)(?:LANGUAGES|IDIOMAS|---|\Z)", t, re.DOTALL
    )
    certs = []
    if cert_m:
        certs = [
            c.strip("- ").strip()
            for c in cert_m[0].split("\n")
            if c.strip().startswith("-")
        ]

    exp_m = re.search(
        r"PROFESSIONAL EXPERIENCE\s*\n\n([^\n]+)\n", t
    )
    current_title = exp_m.group(1).strip() if exp_m else None

    return {
        "skills"       : skills,
        "yr_range"     : yr_range,
        "degree"       : degree,
        "full_degree"  : full_degree,
        "grad_yr"      : grad_yr,
        "certs"        : certs,
        "current_title": current_title,
    }

print("Parsing resumes …")
parsed = [parse_resume(t) for t in df["Resume"]]
df["yr_range"] = [p["yr_range"]      for p in parsed]
df["degree"] = [p["degree"]        for p in parsed]
df["full_degree"] = [p["full_degree"]   for p in parsed]
df["grad_yr"] = [p["grad_yr"]       for p in parsed]
df["current_title"] = [p["current_title"] for p in parsed]
df["resume_len"] = df["Resume"].str.len()
df["jd_len"] = df["Job Description"].str.len()
df["exp_since_grad"]= df["grad_yr"].apply(
    lambda g: 2024 - g if pd.notna(g) and g > 0 else np.nan
)

all_skills = [s for p in parsed for s in p["skills"]]
all_certs = [c for p in parsed for c in p["certs"]]

def get_seniority(text):
    t = str(text).lower()
    senior_kw = ["senior","principal","director","head of","lead","chief","manager"]
    junior_kw = ["junior","intern","assistant","associate","graduate","trainee"]
    if any(k in t for k in senior_kw):
        return "Senior / Lead"
    if any(k in t for k in junior_kw):
        return "Junior / Entry"
    return "Mid-level"

df["seniority"] = df["Resume"].apply(get_seniority)

Parsing resumes …


In [5]:
sep = "─" * 60

print(f"\n{sep}")
print("SECTION 1: DATASET OVERVIEW")
print(sep)
print(f"  Total samples               : {len(df):,}")
print(f"  Features                    : {len(df.columns)}")
print(f"  Missing values              : {df.isnull().sum().sum()}")
print(f"  Positive pairs (Best Match) : {(df['Best Match']==1).sum():,}  "
      f"({(df['Best Match']==1).mean()*100:.1f}%)")
print(f"  Negative pairs              : {(df['Best Match']==0).sum():,}  "
      f"({(df['Best Match']==0).mean()*100:.1f}%)")
print(f"  Unique job roles            : {df['Job Roles'].nunique()}")
print(f"  Job categories              : {df['Category'].nunique()}")

print(f"\n{sep}")
print("SECTION 2: DEMOGRAPHICS")
print(sep)
print(f"  Age range       : {df['Age'].min()} – {df['Age'].max()} years")
print(f"  Mean age        : {df['Age'].mean():.1f}  |  Median: {df['Age'].median():.0f}")
print(f"  Std dev age     : {df['Age'].std():.1f}")
print("\n  Gender distribution:")
for g, c in df["Gender"].value_counts().items():
    print(f"    {g:<8}: {c:,}  ({c/len(df)*100:.1f}%)")
print("\n  Race distribution:")
for r, c in df["Race"].value_counts().items():
    print(f"    {r:<25}: {c:,}  ({c/len(df)*100:.1f}%)")
print("\n  Ethnicities (21 groups):")
for e, c in df["Ethnicity"].value_counts().items():
    print(f"    {e:<20}: {c:,}  ({c/len(df)*100:.1f}%)")

print(f"\n{sep}")
print("SECTION 3: JOB ROLES & CATEGORIES")
print(sep)
print("\n  Category distribution:")
for cat, c in df["Category"].value_counts().items():
    print(f"    {cat:<30}: {c:,}  ({c/len(df)*100:.1f}%)")
print("\n  All 51 Job Roles (count):")
for role, c in df["Job Roles"].value_counts().items():
    print(f"    {role:<40}: {c:,}")

print(f"\n{sep}")
print("SECTION 4: RESUME ANALYSIS")
print(sep)
print(f"\n  Resume text length:")
print(f"    Min    : {df['resume_len'].min():,} chars")
print(f"    Max    : {df['resume_len'].max():,} chars")
print(f"    Mean   : {df['resume_len'].mean():,.0f} chars")
print(f"    Median : {df['resume_len'].median():,.0f} chars")
print(f"\n  Job Description text length:")
print(f"    Min    : {df['jd_len'].min():,} chars")
print(f"    Max    : {df['jd_len'].max():,} chars")
print(f"    Mean   : {df['jd_len'].mean():,.0f} chars")
print(f"    Median : {df['jd_len'].median():,.0f} chars")
print("\n  Experience year ranges (from professional summary):")
yr_c = df["yr_range"].dropna().value_counts()
for yr, c in yr_c.items():
    print(f"    {yr} years : {c:,}  ({c/len(df)*100:.1f}%)")
print("\n  Years since graduation (as of 2024):")
exp_c = df["exp_since_grad"].dropna().astype(int).value_counts().sort_index()
for y, c in exp_c.items():
    print(f"    {y:2d} years : {c:,}")
print(f"\n  Mean years since graduation: {df['exp_since_grad'].mean():.1f}")
print("\n  Degree types:")
for d, c in df["degree"].dropna().value_counts().items():
    print(f"    {d:<12}: {c:,}  ({c/len(df)*100:.1f}%)")
print("\n  Seniority distribution:")
for s, c in df["seniority"].value_counts().items():
    print(f"    {s:<20}: {c:,}  ({c/len(df)*100:.1f}%)")
print(f"\n  Top 30 skills:")
for skill, c in collections.Counter(all_skills).most_common(30):
    print(f"    {skill:<35}: {c:,}")
print(f"\n  Top 15 certifications:")
for cert, c in collections.Counter(all_certs).most_common(15):
    print(f"    {cert[:60]:<60}: {c:,}")

print(f"\n{sep}")
print("SECTION 5: MATCH RATE ANALYSIS")
print(sep)
print("\n  Match rate by gender:")
for g, v in df.groupby("Gender")["Best Match"].mean().items():
    print(f"    {g:<8}: {v*100:.1f}%")
print("\n  Match rate by race:")
for r, v in df.groupby("Race")["Best Match"].mean().items():
    print(f"    {r:<25}: {v*100:.1f}%")
print("\n  Match rate by job category:")
for cat, v in df.groupby("Category")["Best Match"].mean().sort_values(ascending=False).items():
    print(f"    {cat:<30}: {v*100:.1f}%")
print("\n  Match rate by age group:")
df["age_group"] = pd.cut(df["Age"], bins=[24,30,35,40,45,50,56],
                          labels=["25-30","31-35","36-40","41-45","46-50","51-55"])
for ag, v in df.groupby("age_group", observed=True)["Best Match"].mean().items():
    print(f"    {str(ag):<8}: {v*100:.1f}%")


────────────────────────────────────────────────────────────
SECTION 1: DATASET OVERVIEW
────────────────────────────────────────────────────────────
  Total samples               : 10,000
  Features                    : 19
  Missing values              : 6874
  Positive pairs (Best Match) : 4,850  (48.5%)
  Negative pairs              : 5,150  (51.5%)
  Unique job roles            : 51
  Job categories              : 10

────────────────────────────────────────────────────────────
SECTION 2: DEMOGRAPHICS
────────────────────────────────────────────────────────────
  Age range       : 25 – 55 years
  Mean age        : 40.0  |  Median: 40
  Std dev age     : 9.0

  Gender distribution:
    Male    : 5,059  (50.6%)
    Female  : 4,941  (49.4%)

  Race distribution:
    Mongoloid/Asian          : 3,355  (33.6%)
    White/Caucasian          : 3,323  (33.2%)
    Negroid/Black            : 3,322  (33.2%)

  Ethnicities (21 groups):
    Vietnamese          : 506  (5.1%)
    Filipino         

In [6]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Figure 1 — Training Dataset Overview", fontsize=15, fontweight="bold", y=0.98)
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

ax = fig.add_subplot(gs[0, 0])
vals = df["Best Match"].value_counts().sort_index()
bars = ax.bar(["Negative\n(No Match)", "Positive\n(Match)"],
               vals.values, color=["#C0392B", "#2D6A9F"], width=0.55, edgecolor="white")
ax.bar_label(bars, labels=[f"{v:,}\n({v/len(df)*100:.1f}%)" for v in vals.values],
             fontsize=9.5, padding=4)
ax.set_title("Class Distribution")
ax.set_ylabel("Number of Pairs")
ax.set_ylim(0, vals.max() * 1.25)

ax2 = fig.add_subplot(gs[0, 1])
gv = df["Gender"].value_counts()
wedges, texts, autotexts = ax2.pie(
    gv.values, labels=gv.index, autopct="%1.1f%%",
    colors=["#2D6A9F", "#E07B39"], startangle=90,
    textprops={"fontsize": 10}, pctdistance=0.7
)
ax2.set_title("Gender Distribution")

ax3 = fig.add_subplot(gs[0, 2])
rv = df["Race"].value_counts()
short = ["Asian /\nMongoloid", "White /\nCaucasian", "Black /\nNegroid"]
bars3 = ax3.bar(short, rv.values, color=PALETTE[:3], width=0.55, edgecolor="white")
ax3.bar_label(bars3, labels=[f"{v:,}\n({v/len(df)*100:.1f}%)" for v in rv.values],
              fontsize=9, padding=4)
ax3.set_title("Racial Group Distribution")
ax3.set_ylabel("Count")
ax3.set_ylim(0, rv.max() * 1.3)

ax4 = fig.add_subplot(gs[1, :2])
ax4.hist(df["Age"], bins=30, color="#2D6A9F", edgecolor="white", alpha=0.9)
ax4.axvline(df["Age"].mean(), color="#C0392B", linestyle="--", linewidth=1.8,
            label=f"Mean: {df['Age'].mean():.1f}")
ax4.axvline(df["Age"].median(), color="#E07B39", linestyle=":", linewidth=1.8,
            label=f"Median: {df['Age'].median():.0f}")
ax4.set_title("Age Distribution of Applicants")
ax4.set_xlabel("Age (years)")
ax4.set_ylabel("Count")
ax4.legend()

ax5 = fig.add_subplot(gs[1, 2])
eth = df["Ethnicity"].value_counts()
y_pos = range(len(eth))
ax5.barh(list(reversed(list(eth.index))), list(reversed(list(eth.values))),
         color="#3BAA6E", edgecolor="white", height=0.7)
ax5.set_title("Ethnicity Distribution\n(21 Groups)")
ax5.set_xlabel("Count")
ax5.tick_params(axis="y", labelsize=7.5)

plt.savefig("eda_figures/fig1_overview.png")
plt.close()
print("\nSaved: eda_figures/fig1_overview.png")


Saved: eda_figures/fig1_overview.png


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle("Figure 2 — Job Roles and Categories", fontsize=14, fontweight="bold")

roles_sorted = df["Job Roles"].value_counts().sort_values()
colors_role = [PALETTE[i % len(PALETTE)] for i in range(len(roles_sorted))]
axes[0].barh(roles_sorted.index, roles_sorted.values, color=colors_role, height=0.7)
axes[0].set_title("All 51 Job Roles (sample count)")
axes[0].set_xlabel("Number of Samples")
axes[0].tick_params(axis="y", labelsize=7.5)
for i, v in enumerate(roles_sorted.values):
    axes[0].text(v + 1, i, str(v), va="center", fontsize=6.5)

cat_sorted = df["Category"].value_counts().sort_values(ascending=False)
bars = axes[1].bar(cat_sorted.index, cat_sorted.values,
                   color=PALETTE[:len(cat_sorted)], edgecolor="white", width=0.6)
axes[1].bar_label(bars, labels=[f"{v:,}\n({v/len(df)*100:.0f}%)" for v in cat_sorted.values],
                  fontsize=8.5, padding=3)
axes[1].set_title("Job Category Distribution")
axes[1].set_ylabel("Count")
axes[1].set_ylim(0, cat_sorted.max() * 1.25)
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig("eda_figures/fig2_job_roles_categories.png")
plt.close()
print("Saved: eda_figures/fig2_job_roles_categories.png")

Saved: eda_figures/fig2_job_roles_categories.png


In [8]:
fig = plt.figure(figsize=(17, 10))
fig.suptitle("Figure 3 — Education and Professional Experience", fontsize=14, fontweight="bold")
gs = GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.4)

ax = fig.add_subplot(gs[0, 0])
deg_c = df["degree"].dropna().value_counts()
wedges, texts, auts = ax.pie(
    deg_c.values, labels=deg_c.index, autopct="%1.1f%%",
    colors=PALETTE, startangle=140, pctdistance=0.75,
    textprops={"fontsize": 10}
)
ax.set_title("Degree Types")

ax2 = fig.add_subplot(gs[0, 1])
grad_c = df["grad_yr"].dropna().astype(int).value_counts().sort_index()
ax2.bar(grad_c.index, grad_c.values, color="#2D6A9F", edgecolor="white", width=0.8)
ax2.set_title("Graduation Year Distribution")
ax2.set_xlabel("Year")
ax2.set_ylabel("Count")
ax2.set_xticks(grad_c.index[::2])
ax2.tick_params(axis="x", rotation=45)

ax3 = fig.add_subplot(gs[1, 0])
yr_c = df["yr_range"].dropna().value_counts()
yr_order = ["2-4", "3-5", "4-6", "5-7", "6-8", "7-9"]
yr_vals = [yr_c.get(y, 0) for y in yr_order]
bars3 = ax3.bar(yr_order, yr_vals, color=PALETTE[2], edgecolor="white", width=0.6)
ax3.bar_label(bars3, labels=[f"{v:,}" for v in yr_vals], fontsize=9, padding=3)
ax3.set_title("Experience Year Range\n(from Resume Summary)")
ax3.set_xlabel("Years of Experience")
ax3.set_ylabel("Count")
ax3.set_ylim(0, max(yr_vals) * 1.2)

ax4 = fig.add_subplot(gs[1, 1])
sen_c = df["seniority"].value_counts()
colors_s = ["#2D6A9F", "#E07B39", "#3BAA6E"]
bars4 = ax4.bar(sen_c.index, sen_c.values, color=colors_s[:len(sen_c)], edgecolor="white", width=0.55)
ax4.bar_label(bars4, labels=[f"{v:,}\n({v/len(df)*100:.1f}%)" for v in sen_c.values],
              fontsize=9.5, padding=4)
ax4.set_title("Career Seniority Level\n(parsed from resume job titles)")
ax4.set_ylabel("Count")
ax4.set_ylim(0, sen_c.max() * 1.25)

plt.savefig("eda_figures/fig3_education_experience.png")
plt.close()
print("Saved: eda_figures/fig3_education_experience.png")

Saved: eda_figures/fig3_education_experience.png


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle("Figure 4 — Skills and Certifications", fontsize=14, fontweight="bold")

skill_c = collections.Counter(all_skills).most_common(25)
sk_names = [s[0] for s in reversed(skill_c)]
sk_vals = [s[1] for s in reversed(skill_c)]
axes[0].barh(sk_names, sk_vals, color="#2D6A9F", edgecolor="white", height=0.7)
axes[0].set_title("Top 25 Skills Across All Resumes")
axes[0].set_xlabel("Occurrences")
axes[0].tick_params(axis="y", labelsize=8.5)
for i, v in enumerate(sk_vals):
    axes[0].text(v + 3, i, str(v), va="center", fontsize=7.5)

cert_c = collections.Counter(all_certs).most_common(15)
ce_names = [c[0][:45] for c in reversed(cert_c)]
ce_vals = [c[1] for c in reversed(cert_c)]
axes[1].barh(ce_names, ce_vals, color="#3BAA6E", edgecolor="white", height=0.7)
axes[1].set_title("Top 15 Professional Certifications")
axes[1].set_xlabel("Occurrences")
axes[1].tick_params(axis="y", labelsize=8)
for i, v in enumerate(ce_vals):
    axes[1].text(v + 0.5, i, str(v), va="center", fontsize=8)

plt.tight_layout()
plt.savefig("eda_figures/fig4_skills_certifications.png")
plt.close()
print("Saved: eda_figures/fig4_skills_certifications.png")

Saved: eda_figures/fig4_skills_certifications.png


In [10]:
fig = plt.figure(figsize=(17, 10))
fig.suptitle("Figure 5 — Match Rate Analysis (Best Match = 1)", fontsize=14, fontweight="bold")
gs = GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.38)

def pct_fmt(ax):
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    ax.set_ylabel("Match Rate (%)")
    ax.set_ylim(0, 1.05)

ax = fig.add_subplot(gs[0, 0])
mr_g = df.groupby("Gender")["Best Match"].mean().sort_values(ascending=False)
bars = ax.bar(mr_g.index, mr_g.values, color=["#2D6A9F","#E07B39"], edgecolor="white", width=0.5)
ax.bar_label(bars, labels=[f"{v*100:.1f}%" for v in mr_g.values], fontsize=11, padding=4)
ax.set_title("Match Rate by Gender")
pct_fmt(ax)

ax2 = fig.add_subplot(gs[0, 1])
mr_r = df.groupby("Race")["Best Match"].mean().sort_values(ascending=False)
short = ["Asian /\nMongoloid", "Black /\nNegroid", "White /\nCaucasian"]
bars2 = ax2.bar(short[:len(mr_r)], mr_r.values, color=PALETTE[:3], edgecolor="white", width=0.5)
ax2.bar_label(bars2, labels=[f"{v*100:.1f}%" for v in mr_r.values], fontsize=11, padding=4)
ax2.set_title("Match Rate by Race")
pct_fmt(ax2)

ax3 = fig.add_subplot(gs[1, :])
mr_cat = df.groupby("Category")["Best Match"].mean().sort_values(ascending=False)
bars3 = ax3.bar(mr_cat.index, mr_cat.values,
                 color=PALETTE[:len(mr_cat)], edgecolor="white", width=0.6)
ax3.bar_label(bars3, labels=[f"{v*100:.1f}%" for v in mr_cat.values],
              fontsize=9.5, padding=4)
ax3.set_title("Match Rate by Job Category")
pct_fmt(ax3)
ax3.tick_params(axis="x", rotation=20)
ax3.axhline(df["Best Match"].mean(), linestyle="--", color="grey", linewidth=1.3,
            label=f"Overall mean: {df['Best Match'].mean()*100:.1f}%")
ax3.legend(fontsize=9)

plt.savefig("eda_figures/fig5_match_rate_analysis.png")
plt.close()
print("Saved: eda_figures/fig5_match_rate_analysis.png")

Saved: eda_figures/fig5_match_rate_analysis.png


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 6 — Document Length Distributions", fontsize=14, fontweight="bold")

axes[0].hist(df["resume_len"], bins=50, color="#2D6A9F", edgecolor="white", alpha=0.9)
axes[0].axvline(df["resume_len"].mean(), color="#C0392B", linestyle="--", linewidth=1.8,
                label=f"Mean: {df['resume_len'].mean():,.0f} chars")
axes[0].set_title("Resume Text Length")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(df["jd_len"], bins=50, color="#3BAA6E", edgecolor="white", alpha=0.9)
axes[1].axvline(df["jd_len"].mean(), color="#C0392B", linestyle="--", linewidth=1.8,
                label=f"Mean: {df['jd_len'].mean():,.0f} chars")
axes[1].set_title("Job Description Text Length")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_figures/fig6_text_lengths.png")
plt.close()
print("Saved: eda_figures/fig6_text_lengths.png")

Saved: eda_figures/fig6_text_lengths.png


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 7 — Match Rate by Age and Experience", fontsize=14, fontweight="bold")

mr_age = df.groupby("age_group", observed=True)["Best Match"].mean()
bars = axes[0].bar(mr_age.index.astype(str), mr_age.values,
                     color=PALETTE, edgecolor="white", width=0.6)
axes[0].bar_label(bars, labels=[f"{v*100:.1f}%" for v in mr_age.values],
                  fontsize=10, padding=3)
axes[0].set_title("Match Rate by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
axes[0].set_ylabel("Match Rate")
axes[0].set_ylim(0, 1.0)

yr_order = ["2-4", "3-5", "4-6", "5-7", "6-8", "7-9"]
yr_c2 = df[df["yr_range"].notna()].groupby("yr_range")["Best Match"].mean()
yr_vals2 = [yr_c2.get(y, 0) for y in yr_order if y in yr_c2.index]
yr_lbls = [y for y in yr_order if y in yr_c2.index]
bars2 = axes[1].bar(yr_lbls, yr_vals2, color="#2D6A9F", edgecolor="white", width=0.6)
axes[1].bar_label(bars2, labels=[f"{v*100:.1f}%" for v in yr_vals2],
                  fontsize=10, padding=3)
axes[1].set_title("Match Rate by Experience Range\n(years in resume summary)")
axes[1].set_xlabel("Years of Experience")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
axes[1].set_ylabel("Match Rate")
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig("eda_figures/fig7_match_age_experience.png")
plt.close()
print("Saved: eda_figures/fig7_match_age_experience.png")

Saved: eda_figures/fig7_match_age_experience.png


In [13]:
top_skills = [s for s, _ in collections.Counter(all_skills).most_common(12)]
cat_list = df["Category"].unique().tolist()

def skill_rate(group_df, skill):
    return group_df["Resume"].str.contains(skill, case=False, na=False).mean()

matrix = np.array([
    [skill_rate(df[df["Category"] == cat], sk) for sk in top_skills]
    for cat in cat_list
])

fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(matrix, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(top_skills)))
ax.set_xticklabels(top_skills, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(cat_list)))
ax.set_yticklabels(cat_list, fontsize=9)
plt.colorbar(im, ax=ax, label="Proportion of resumes containing skill")
for i in range(len(cat_list)):
    for j in range(len(top_skills)):
        ax.text(j, i, f"{matrix[i,j]*100:.0f}%", ha="center", va="center",
                fontsize=7.5, color="white" if matrix[i,j] > 0.4 else "black")
ax.set_title("Figure 8 — Top Skills Presence by Job Category", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_figures/fig8_skills_heatmap.png")
plt.close()
print("Saved: eda_figures/fig8_skills_heatmap.png")

Saved: eda_figures/fig8_skills_heatmap.png


In [14]:
fig, ax = plt.subplots(figsize=(13, 6))
deg_cat = pd.crosstab(df["Category"], df["degree"], normalize="index")
deg_cat_plot = deg_cat[["Bachelor", "Master", "Doctor", "Associate"]].fillna(0)
bottom = np.zeros(len(deg_cat_plot))
deg_colors = {"Bachelor": "#2D6A9F", "Master": "#3BAA6E",
              "Doctor": "#E07B39", "Associate": "#C0392B"}
for deg in deg_cat_plot.columns:
    ax.bar(deg_cat_plot.index, deg_cat_plot[deg].values, bottom=bottom,
           label=deg, color=deg_colors.get(deg, "grey"), edgecolor="white", width=0.65)
    bottom += deg_cat_plot[deg].values
ax.set_title("Figure 9 — Degree Level Distribution by Job Category", fontsize=13, fontweight="bold")
ax.set_ylabel("Proportion")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.tick_params(axis="x", rotation=22)
ax.legend(title="Degree Type", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("eda_figures/fig9_degree_by_category.png")
plt.close()
print("Saved: eda_figures/fig9_degree_by_category.png")

Saved: eda_figures/fig9_degree_by_category.png


In [15]:
fig = plt.figure(figsize=(12, 6))
fig.suptitle("Figure 10 — Training Dataset: Key Statistics at a Glance",
             fontsize=14, fontweight="bold")
ax = fig.add_subplot(111)
ax.axis("off")

stats = [
    ["Total samples",           f"{len(df):,}"],
    ["Unique job roles",         "51"],
    ["Job categories",           "10"],
    ["Positive pairs",          f"{(df['Best Match']==1).sum():,} ({(df['Best Match']==1).mean()*100:.1f}%)"],
    ["Negative pairs",          f"{(df['Best Match']==0).sum():,} ({(df['Best Match']==0).mean()*100:.1f}%)"],
    ["Gender balance",          f"Male {(df['Gender']=='Male').mean()*100:.1f}% / Female {(df['Gender']=='Female').mean()*100:.1f}%"],
    ["Racial groups",           "3 (Asian/Mongoloid, White/Caucasian, Black/Negroid)"],
    ["Ethnicities",             "21 (Vietnamese, Filipino, Chinese, Irish, Kenyan …)"],
    ["Age range",               f"{df['Age'].min()}–{df['Age'].max()} yrs (mean {df['Age'].mean():.1f})"],
    ["Degree types",            "Bachelor, Master, Doctor/PhD, Associate, Juris Doctor"],
    ["Avg resume length",       f"{df['resume_len'].mean():,.0f} characters"],
    ["Avg job desc. length",    f"{df['jd_len'].mean():,.0f} characters"],
    ["Most common skills",      "Problem Solving, Project Management, Data Analysis"],
    ["Experience range",        "2–9 years (stated in resume summary)"],
    ["Missing values",          "None"],
]

table = ax.table(
    cellText=stats,
    colLabels=["Statistic", "Value"],
    cellLoc="left",
    loc="center",
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("lightgrey")
    if row == 0:
        cell.set_facecolor("#2D6A9F")
        cell.set_text_props(color="white", fontweight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#EEF4FA")
    else:
        cell.set_facecolor("white")
    cell.set_height(0.062)

plt.savefig("eda_figures/fig10_summary_card.png")
plt.close()
print("Saved: eda_figures/fig10_summary_card.png")

print(f"\n{'─'*60}")
print("All 10 figures saved to ./eda_figures/")
print("EDA complete.")

Saved: eda_figures/fig10_summary_card.png

────────────────────────────────────────────────────────────
All 10 figures saved to ./eda_figures/
EDA complete.
